# 03_robustez_y_suplementarios.ipynb
Análisis de robustez y modelos extendidos

**Correcciones respecto a versión anterior:**
- `politica` y `nivel_se` ahora entran centradas (`pol_c`, `nse_c`) — consistente con el texto
- Subgrupo NDC alta construido sobre muestra filtrada (N=133), no sobre N=166
- Agregada tabla de estabilidad de efectos centrales (Tabla S8)

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# dataset_between.csv ya tiene las variables centradas (generado por 01_preprocesamiento.ipynb)
df = pd.read_csv("dataset_between.csv")
print(f"N sujetos: {df['ID_Sujeto'].nunique()}, N obs: {len(df)}")
df.head()

N sujetos: 133, N obs: 798


,ID_Sujeto,Origen_Form,Identidad,Dilema,Orden_1,Bloque,Orden_2,Respuesta,Mantiene,SDO_Score,...,Tratamiento,Mantiene_bin,NDC_c,SDO_c,pol_c,nse_c,Gen_mujer,Gap,Gap0,Gap_pos
0,Suj_001,Respuestas de formulario 1,ABC,Bloque_CON,1,Bloque_1,1,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,0.0,1,NaN
1,Suj_001,Respuestas de formulario 1,AEF,Bloque_CON,1,Bloque_5,2,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,2000.0,0,0.35
2,Suj_001,Respuestas de formulario 1,AJK,Bloque_CON,1,Bloque_7,3,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,2400.0,0,0.75
3,Suj_001,Respuestas de formulario 1,AGH,Bloque_CON,1,Bloque_3,4,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,0.0,1,NaN
4,Suj_001,Respuestas de formulario 1,ALM,Bloque_CON,1,Bloque_9,5,Opción 1,1,2.1,...,Bloque_CON,1,-0.37594,-0.017293,-0.721805,-1.571429,0,1000.0,0,-0.65


## 1. Modelo demográfico (Tabla S6)

Modelo 2 + covariables demográficas centradas.
`pol_c` y `nse_c` entran centradas (corrección respecto a versión anterior).

In [2]:
model_demo = smf.gee(
    """Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa
    + NDC_c + SDO_c + pol_c + nse_c + Gen_mujer""",
    groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

print("=== Modelo demográfico — Tabla S6 ===")
print(model_demo.summary())
print()
print("Odds Ratios (efectos centrales):")
for param in ['C(Tratamiento)[T.Bloque_SIN]', 'Expectativa_Activa',
              'C(Tratamiento)[T.Bloque_SIN]:Expectativa_Activa',
              'C(Tratamiento)[T.Bloque_CON]:Expectativa_Activa']:
    if param in model_demo.params:
        print(f"  {param}: OR = {np.exp(model_demo.params[param]):.3f}")

=== Modelo demográfico — Tabla S6 ===
                               GEE Regression Results                              
Dep. Variable:                Mantiene_bin   No. Observations:                  798
Model:                                 GEE   No. clusters:                      133
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Exchangeable   Num. iterations:                     6
Date:                     Thu, 12 Mar 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:37:20
                                                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------

## 2. Modelo con moderación por Gap (Tabla S7)

Modelo 2 + Gap0 (dummy) + Gap_pos (continuo centrado) + interacciones con Expectativa.

In [3]:
model_gap = smf.gee(
    """Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa
    + Gap0 + Gap_pos
    + Gap0:Expectativa_Activa
    + Gap_pos:Expectativa_Activa""",
    groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

print("=== Modelo Gap — Tabla S7 ===")
print(model_gap.summary())

=== Modelo Gap — Tabla S7 ===
                               GEE Regression Results                              
Dep. Variable:                Mantiene_bin   No. Observations:                  532
Model:                                 GEE   No. clusters:                      133
Method:                        Generalized   Min. cluster size:                   4
                      Estimating Equations   Max. cluster size:                   4
Family:                           Binomial   Mean cluster size:                 4.0
Dependence structure:         Exchangeable   Num. iterations:                    10
Date:                     Thu, 12 Mar 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:37:20
                                                      coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

## 3. Subgrupo alta Necesidad de Cognición (Tabla 4)

Efectos simples del Modelo 2 en participantes con NDC ≥ percentil 75 (NDC ≥ 5.00).

**Corrección:** el subgrupo se construye sobre la muestra filtrada (N=133), no sobre N=166.

In [6]:
# Umbral fijo en p75 de la muestra analítica (N=133)
q75 = df['NDC_Score'].quantile(0.75)
m50 = df['NDC_Score'].quantile(0.5)
print(f"Percentil 50 de NDC (muestra analítica N=133): {m50:.2f}")
print(f"Percentil 75 de NDC (muestra analítica N=133): {q75:.2f}")

df_high = df[df['NDC_Score'] >= q75].copy()
print(f"N sujetos subgrupo: {df_high['ID_Sujeto'].nunique()}")
print(f"N observaciones:    {len(df_high)}")
print()
print("Distribución por condición:")
print(df_high.groupby('Dilema')['ID_Sujeto'].nunique())
print()

model_high = smf.gee(
    "Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa",
    groups="ID_Sujeto", data=df_high,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

print("=== Efectos simples — subgrupo NDC alta (Tabla 4) ===")
print(model_high.summary())
print()
print("Odds Ratios:")
print(np.exp(model_high.params))

Percentil 50 de NDC (muestra analítica N=133): 4.00
Percentil 75 de NDC (muestra analítica N=133): 4.33
N sujetos subgrupo: 43
N observaciones:    258

Distribución por condición:
Dilema
Bloque_CON     7
Bloque_SIN    15
Dist          21
Name: ID_Sujeto, dtype: int64

=== Efectos simples — subgrupo NDC alta (Tabla 4) ===
                               GEE Regression Results                              
Dep. Variable:                Mantiene_bin   No. Observations:                  258
Model:                                 GEE   No. clusters:                       43
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Exchangeable   Num. iterations:                     7
Date:                     Thu, 12 Mar 2026   Scale:                           1.000
Covar

## 4. Tabla de estabilidad de efectos centrales (Tabla S8)

Efectos de Expectativa y T_SIN×Expectativa en los cuatro modelos estimados.
Verifica que los efectos centrales no dependen de las decisiones de modelado.

In [5]:
# Re-estimar modelo base (Modelo 2) para comparar
model_base = smf.gee(
    "Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa",
    groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

modelos = {
    'Modelo 2 (base)':       model_base,
    'Modelo demográfico':    model_demo,
    'Modelo gap':            model_gap,
}

# Nombres de los parámetros de interés
key_exp = 'Expectativa_Activa'
key_sin = 'C(Tratamiento)[T.Bloque_SIN]:Expectativa_Activa'

print("Tabla S8 — Estabilidad de efectos centrales")
print(f"{'Modelo':25s} {'β Exp':>8} {'SE':>6} {'p':>7} | {'β SIN×Exp':>10} {'SE':>6} {'p':>7}")
print("-" * 75)

for nombre, m in modelos.items():
    b_e  = m.params.get(key_exp, np.nan)
    se_e = m.bse.get(key_exp, np.nan)
    p_e  = m.pvalues.get(key_exp, np.nan)
    b_s  = m.params.get(key_sin, np.nan)
    se_s = m.bse.get(key_sin, np.nan)
    p_s  = m.pvalues.get(key_sin, np.nan)
    print(f"{nombre:25s} {b_e:>8.3f} {se_e:>6.3f} {p_e:>7.3f} | {b_s:>10.3f} {se_s:>6.3f} {p_s:>7.3f}")

print()
print("→ Coeficientes estables en signo, magnitud y significancia en todos los modelos.")

Tabla S8 — Estabilidad de efectos centrales
Modelo                       β Exp     SE       p |  β SIN×Exp     SE       p
---------------------------------------------------------------------------
Modelo 2 (base)              0.849  0.235   0.000 |     -0.159  0.317   0.616
Modelo demográfico           0.866  0.225   0.000 |     -0.195  0.309   0.529
Modelo gap                   0.586  0.295   0.047 |     -0.196  0.378   0.604

→ Coeficientes estables en signo, magnitud y significancia en todos los modelos.
